# Failure Cases & Limitations (LIVE DEMO — Optional)

**Purpose:** Show failure modes so students don’t assume semantic search is always correct.

> **Similarity ≠ correctness**  
> **Similarity ≠ usefulness**  
> **Retrieval ≠ decision-making**

In this notebook we’ll demo common failure modes:
1. **Ambiguity** (one query, multiple intents)
2. **Domain mismatch** (corpus vs query language/style)
3. **Semantic but not task-relevant** (sounds related but doesn’t help)
4. **Bias / representational issues** (embeddings reflect training data)

Then we’ll cover what to do about it:
- better data
- better encoder
- re-ranking
- metadata filters
- evaluation


## 0) Setup

We’ll build:
- a small corpus of snippets (with multiple domains)
- embeddings + FAISS index
- a helper search function


In [1]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

In [2]:
import re
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 150)

## 1) Build a demo corpus

We intentionally mix topics and styles so we can trigger realistic failure modes.
We also add a small set of **academic-style** snippets to demonstrate domain mismatch.


In [3]:
docs = [
    # Finance/business
    ("finance", "Banks assess credit risk before approving loans."),
    ("finance", "Reduce spending by reviewing subscriptions and recurring bills."),
    ("finance", "High-interest debt costs more over time than low-interest debt."),
    ("business", "Customer retention improves when support resolves issues quickly."),
    ("business", "A loyalty program can increase repeat purchases."),
    ("business", "Segment users to tailor messaging to their needs."),

    # Software/data
    ("software", "Refactor code to reduce technical debt and improve maintainability."),
    ("software", "Profile your program to find performance bottlenecks."),
    ("software", "Add caching to avoid recomputing expensive results."),
    ("data", "Use indexes in databases to speed up query execution."),
    ("data", "Validate datasets by checking missing values and outliers."),
    ("data", "A confusion matrix summarizes classification errors."),

    # Weather/safety
    ("weather", "Monitor official advisories when a typhoon is nearby."),
    ("weather", "Avoid driving through flooded roads during heavy rain."),
    ("weather", "Storm surge can be more dangerous than wind in coastal zones."),

    # Health
    ("health", "Breathing exercises can reduce stress in the short term."),
    ("health", "Prioritize sleep to support focus and memory."),
    ("health", "Walking daily can improve cardiovascular health."),

    # Food
    ("food", "Simmer soup slowly to deepen the flavor."),
    ("food", "Try adding chili oil to ramen for extra spice."),

    # Academic-ish (domain mismatch examples)
    ("academic", "We evaluate semantic similarity using cosine distance in embedding space."),
    ("academic", "The encoder maps text into a dense vector representation."),
    ("academic", "Approximate nearest neighbor search improves retrieval latency at scale."),
    ("academic", "Ablation studies isolate the effect of each model component."),
    ("academic", "The paper reports statistically significant improvements over baselines."),
]

df = pd.DataFrame(docs, columns=["topic", "text"])
df.insert(0, "id", [f"D{i:03d}" for i in range(len(df))])
df

,id,topic,text
0,D000,finance,Banks assess credit risk before approving loans.
1,D001,finance,Reduce spending by reviewing subscriptions and recurring bills.
2,D002,finance,High-interest debt costs more over time than low-interest debt.
3,D003,business,Customer retention improves when support resolves issues quickly.
4,D004,business,A loyalty program can increase repeat purchases.
5,D005,business,Segment users to tailor messaging to their needs.
6,D006,software,Refactor code to reduce technical debt and improve maintainability.
7,D007,software,Profile your program to find performance bottlenecks.
8,D008,software,Add caching to avoid recomputing expensive results.
9,D009,data,Use indexes in databases to speed up query execution.


## 2) Build embeddings + FAISS index

We’ll use normalized embeddings + inner product index (cosine-like).


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

texts = df["text"].tolist()
emb = model.encode(texts, normalize_embeddings=True).astype("float32")

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Embeddings:", emb.shape, "| Index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings: (25, 384) | Index size: 25


## 3) Search helper

We’ll show:
- score (cosine-ish)
- topic
- text

We’ll also show a quick token overlap count (not a keyword baseline — just a hint).


In [5]:
def _tokenize(s: str):
    return set(re.findall(r"[a-zA-Z]+", s.lower()))

def search(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    q_tokens = _tokenize(query)
    
    rows = []
    for rank, (i, score) in enumerate(zip(idx[0], scores[0]), start=1):
        text = df.loc[i, "text"]
        overlap = len(q_tokens & _tokenize(text))
        rows.append({
            "rank": rank,
            "id": df.loc[i, "id"],
            "topic": df.loc[i, "topic"],
            "score(cos≈ip)": float(score),
            "overlap_tokens": overlap,
            "text": text,
        })
    return pd.DataFrame(rows)

# quick sanity check
search("How do banks decide loans?", k=5)

,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D000,finance,0.649603,2,Banks assess credit risk before approving loans.
1,2,D002,finance,0.347536,0,High-interest debt costs more over time than low-interest debt.
2,3,D006,software,0.204996,0,Refactor code to reduce technical debt and improve maintainability.
3,4,D001,finance,0.170690,0,Reduce spending by reviewing subscriptions and recurring bills.
4,5,D003,business,0.166603,0,Customer retention improves when support resolves issues quickly.


# Failure Mode 1: Ambiguity

**One query, multiple intents.**

Example: “*index*” can mean:
- database index (data engineering)
- FAISS / ANN index (vector search)
- stock index (finance)

The embedding may retrieve “reasonable” matches, but not necessarily the intent the user meant.


In [6]:
ambiguous_query = "How do I build an index to make search faster?"
search(ambiguous_query, k=8)

,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D009,data,0.705909,1,Use indexes in databases to speed up query execution.
1,2,D022,academic,0.516789,1,Approximate nearest neighbor search improves retrieval latency at scale.
2,3,D008,software,0.447711,1,Add caching to avoid recomputing expensive results.
3,4,D007,software,0.282105,1,Profile your program to find performance bottlenecks.
4,5,D020,academic,0.199148,0,We evaluate semantic similarity using cosine distance in embedding space.
5,6,D016,health,0.187707,1,Prioritize sleep to support focus and memory.
6,7,D018,food,0.158406,1,Simmer soup slowly to deepen the flavor.
7,8,D001,finance,0.149374,0,Reduce spending by reviewing subscriptions and recurring bills.


### Commentary: why this happens

- The query contains the word **“index”** and the concept **“faster search.”**
- Multiple topics in the corpus match parts of that meaning:
  - database indexes
  - ANN indexes
  - semantic similarity / embeddings

**Takeaway:** Semantic search can’t read minds.
It guesses intent from the query + corpus.


# Failure Mode 2: Domain mismatch

**Corpus is academic; query is slang.**

If a user speaks in slang, shorthand, or local jargon, and your corpus is formal/academic,
retrieval can become noisy or misleading.

We’ll try a slangy query and see what it pulls.


In [7]:
domain_mismatch_query = "bro my code is hella laggy, how do i make it not trash"
search(domain_mismatch_query, k=8)

,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D006,software,0.326617,1,Refactor code to reduce technical debt and improve maintainability.
1,2,D008,software,0.293009,0,Add caching to avoid recomputing expensive results.
2,3,D007,software,0.246409,0,Profile your program to find performance bottlenecks.
3,4,D018,food,0.187231,0,Simmer soup slowly to deepen the flavor.
4,5,D001,finance,0.174081,0,Reduce spending by reviewing subscriptions and recurring bills.
5,6,D009,data,0.158483,0,Use indexes in databases to speed up query execution.
6,7,D021,academic,0.149356,0,The encoder maps text into a dense vector representation.
7,8,D016,health,0.141923,0,Prioritize sleep to support focus and memory.


### Commentary: why this happens

- Slang terms (“bro”, “hella”, “trash”) are not in many corpora.
- The encoder may still catch “code” + “laggy” → performance issues,
  but the match quality depends on the model’s exposure to such language.

**Takeaway:** The embedding model and your corpus need to reflect your users’ language.


# Failure Mode 3: Semantic but not task-relevant

Sometimes results are **semantically similar** but **don’t help the decision** the user needs.

Example: user wants an *actionable checklist* for typhoon preparation, but retrieval returns:
- general facts
- definitions
- adjacent concepts

Let’s try a query that demands actionable guidance.


In [8]:
not_task_relevant_query = "What should I do tonight to prepare for a typhoon hitting tomorrow? Give me a checklist."
search(not_task_relevant_query, k=8)

,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D012,weather,0.616836,2,Monitor official advisories when a typhoon is nearby.
1,2,D013,weather,0.337836,0,Avoid driving through flooded roads during heavy rain.
2,3,D014,weather,0.315259,0,Storm surge can be more dangerous than wind in coastal zones.
3,4,D019,food,0.218401,2,Try adding chili oil to ramen for extra spice.
4,5,D016,health,0.215205,1,Prioritize sleep to support focus and memory.
5,6,D001,finance,0.172357,0,Reduce spending by reviewing subscriptions and recurring bills.
6,7,D006,software,0.167941,1,Refactor code to reduce technical debt and improve maintainability.
7,8,D011,data,0.138774,1,A confusion matrix summarizes classification errors.


### Commentary: why this happens

- The query is about **actions** (checklist), but the corpus might contain mostly **facts**.
- Semantic similarity retrieves “typhoon”, “storm surge”, “advisories” — related, but not a checklist.

**Takeaway:** Retrieval quality depends on whether your corpus contains the *kind* of content you want to serve.


# Failure Mode 4: Bias / representational issues

Embeddings reflect:
- what the model was trained on
- what your corpus contains
- which groups/topics are over- or under-represented

This can show up as:
- missing relevant results for certain phrasing
- overly generic results for underrepresented concepts
- skewed associations

We can’t fully “prove bias” in a tiny demo, but we can observe symptoms.
Try multiple phrasings and see what changes.


In [9]:
bias_probe_queries = [
    "How do I manage stress quickly?",
    "I'm panicking—what can I do right now?",
    "I feel burnt out at work; what should I do?",
]

for q in bias_probe_queries:
    print("="*95)
    print("QUERY:", q)
    display(search(q, k=6))

QUERY: How do I manage stress quickly?


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D015,health,0.566143,1,Breathing exercises can reduce stress in the short term.
1,2,D016,health,0.431644,0,Prioritize sleep to support focus and memory.
2,3,D018,food,0.292854,0,Simmer soup slowly to deepen the flavor.
3,4,D001,finance,0.183189,0,Reduce spending by reviewing subscriptions and recurring bills.
4,5,D009,data,0.181939,0,Use indexes in databases to speed up query execution.
5,6,D019,food,0.163139,0,Try adding chili oil to ramen for extra spice.


QUERY: I'm panicking—what can I do right now?


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D015,health,0.277539,1,Breathing exercises can reduce stress in the short term.
1,2,D016,health,0.234844,0,Prioritize sleep to support focus and memory.
2,3,D018,food,0.213300,0,Simmer soup slowly to deepen the flavor.
3,4,D013,weather,0.149965,0,Avoid driving through flooded roads during heavy rain.
4,5,D019,food,0.145251,0,Try adding chili oil to ramen for extra spice.
5,6,D014,weather,0.137996,1,Storm surge can be more dangerous than wind in coastal zones.


QUERY: I feel burnt out at work; what should I do?


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D015,health,0.226666,0,Breathing exercises can reduce stress in the short term.
1,2,D016,health,0.224330,0,Prioritize sleep to support focus and memory.
2,3,D019,food,0.203253,0,Try adding chili oil to ramen for extra spice.
3,4,D008,software,0.172918,0,Add caching to avoid recomputing expensive results.
4,5,D001,finance,0.108220,0,Reduce spending by reviewing subscriptions and recurring bills.
5,6,D003,business,0.108085,0,Customer retention improves when support resolves issues quickly.


### Commentary: what to look for

- Do different phrasings retrieve different “kinds” of advice?
- Are some user styles (formal vs emotional) handled better?
- Are results overly generic for certain phrasing?

**Takeaway:** This is why you evaluate retrieval on *real user queries*.


# What to do about it (Mitigations)

## 1) Better data (usually the biggest win)
- Add more examples of what users actually ask
- Add the content you want to serve (checklists, SOPs, playbooks)
- Add domain-specific terminology and style variations

## 2) Better encoder
- Try newer or domain-tuned embedding models
- Consider multilingual or local-language models if needed

## 3) Re-ranking
- Retrieve top 50 by embeddings, then re-rank with:
  - a cross-encoder
  - an LLM judge (carefully)
  - heuristic scoring (freshness, authority)

## 4) Metadata filters
- Filter by domain/topic before similarity search (or after)
- Apply constraints like geography, date, source reliability, permissions

## 5) Evaluation
- Build a small test set of queries → expected relevant docs
- Track metrics like Recall@k, MRR, nDCG
- Continuously log real queries + user feedback


## Outputs checklist

- ✅ “Bad” retrieval examples for each failure mode
- ✅ Short commentary explaining why it happened
- ✅ Practical mitigation strategies


## Live demo script (optional)

1. Run Ambiguity example and ask: **“What did the user mean by index?”**
2. Run Domain mismatch and ask: **“Would our real users talk like this?”**
3. Run Not task-relevant and ask: **“Do we have the right content?”**
4. Run Bias probe and ask: **“Which phrasing does the system handle best?”**
5. End with: **“So what do we do about it?”** → mitigations slide.
